# Feature audit and experiment comparison
Outputs below inspect executed experiments and real historical feature rows. Model development lives in tested Python modules.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'backend').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from backend.database import make_engine
import matplotlib.pyplot as plt


In [2]:
from backend.ml.features import source_frame, build_features, FEATURE_SETS
frame=build_features(source_frame(make_engine()))
frame.loc[(frame.year==2024)&(frame.driver_id==844),['date','race_id',*FEATURE_SETS['full']]].head()

,date,race_id,grid_position,driver_recent_podium,driver_recent_finish,driver_recent_dnf,constructor_recent_podium,driver_circuit_podium,driver_career_podium
13862,2024-03-02,1121,2.0,0.6,9.4,0.4,0.4,0.245000,0.233333
13881,2024-03-09,1122,2.0,0.6,6.2,0.2,0.4,0.241667,0.231618
13902,2024-03-24,1123,4.0,0.6,6.2,0.2,0.4,0.207143,0.237226
13921,2024-04-07,1124,8.0,0.8,2.6,0.0,0.6,0.207143,0.242754
13941,2024-04-21,1125,6.0,0.6,3.0,0.0,0.6,0.090000,0.241007


In [3]:
comparison=pd.read_csv(ROOT/'reports/model_comparison.csv')
comparison[['experiment_id','model','feature_set','log_loss','brier_score','top3_hit_rate']]

,experiment_id,model,feature_set,log_loss,brier_score,top3_hit_rate
0,EXP-001,StandardScaler + LogisticRegression,grid,0.246224,0.069488,0.716667
1,EXP-002,logistic,driver,0.227574,0.066091,0.683333
2,EXP-003,logistic,team,0.226373,0.065867,0.700000
3,EXP-004,logistic,full,0.223892,0.065399,0.677778
4,EXP-005,logistic,career,0.228719,0.064872,0.700000
5,EXP-006,forest,full,0.214612,0.062845,0.722222
6,EXP-007,boosting,full,0.209820,0.062444,0.727778
7,EXP-008,logistic,full,0.224788,0.065712,0.677778
8,EXP-009,boosting,full,0.211740,0.062898,0.700000


In [4]:
ablations=json.loads((ROOT/'reports/hypothesis_ablations.json').read_text())
pd.DataFrame([{'experiment':r['experiment_id'],'validation_log_loss':r['validation']['log_loss'],'validation_brier':r['validation']['brier_score']} for r in ablations])

,experiment,validation_log_loss,validation_brier
0,H3_recent,0.229013,0.065958
1,H3_career,0.228719,0.064872
2,H4_without_circuit,0.226373,0.065867
3,H4_with_circuit,0.225183,0.065440


Recent podium rate alone did not outperform career rate. Circuit history modestly improved the controlled team-feature comparison. These are descriptive validation comparisons, not causal claims or significance tests.